# Reward-Conditioned Neural Memory Layer — Full 10-Session Evaluation

**Purpose:** Run the definitive single-run evaluation with 10 sessions to demonstrate stable memory accumulation.  
**Runtime:** GPU (T4 16 GB), ~4–5 hours total (10 sessions × 16 problems).  
**Success criteria:** improvement-on-failed-problems > 40% by session 10, with monotone-trending pass rate.

| Cell | Step | Gate |
|---|---|---|
| 1 | Install deps + create dirs | — |
| 2 | Copy src files from dataset | All 5 files present |
| 3 | Isolation tests (CPU) | All 5 must PASS |
| 4 | Load Gemma 2B | Model loads on GPU |
| 5 | Inject memory + perplexity check | Delta < 1% |
| 6 | Download + calibrate HumanEval | ≥10 problems in 30–70% range |
| 7 | Full 10-session evaluation | improvement-on-failed > 40% |
| 8 | Zip + download checkpoint | Before session expires |

In [ ]:
# ── CELL 1: Install dependencies + create directory structure ──
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "accelerate", "datasets", "faiss-cpu"],
    check=True
)

import os, json, torch
from pathlib import Path

sys.path.insert(0, "/kaggle/working/src")

for d in ["src", "problems", "memory_states", "results", "logs"]:
    Path(f"/kaggle/working/{d}").mkdir(parents=True, exist_ok=True)

print("Env ready. CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ── CELL 2: Copy source files from uploaded Kaggle dataset ──
import shutil

SOURCE = "/kaggle/input/datasets/atromixtheanalyzer/memory-agent/src"
DEST   = "/kaggle/working/src"

if os.path.exists(SOURCE):
    for f in os.listdir(SOURCE):
        shutil.copy(os.path.join(SOURCE, f), os.path.join(DEST, f))
    print("Source files copied:", os.listdir(DEST))
else:
    print(f"WARNING: {SOURCE} not found.")
    print("Upload the src/ folder as a Kaggle dataset, or copy files manually.")
    print("Required: memory_module.py, model_surgery.py, sandbox.py,")
    print("          session_manager.py, eval.py")

In [ ]:
# ── CELL 3: Isolation tests (CPU — no GPU needed) ──
# ALL 5 tests must PASS before loading the LLM.

from memory_module import run_all_isolation_tests
run_all_isolation_tests()

In [ ]:
# ── CELL 4: Load Gemma 2B ──
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")
MODEL_ID = "google/gemma-2-2b-it"

login(token=HF_TOKEN, add_to_git_credential=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded. Device:", next(model.parameters()).device)

In [ ]:
# ── CELL 5: Inject memory layers + perplexity gate check ──
from model_surgery import inject_memory_layers, measure_perplexity

VALIDATION_TEXTS = [
    "def add(a, b):\n    return a + b",
    "def reverse(s):\n    return s[::-1]",
    "for i in range(10):\n    print(i)",
    "x = [1, 2, 3]\nx.sort()\nprint(x)",
    "def is_prime(n):\n    return n > 1 and all(n % i for i in range(2, n))",
    "import os\nprint(os.getcwd())",
    "d = {'a': 1, 'b': 2}\nprint(d.get('a', 0))",
    "result = [x**2 for x in range(5)]",
    "def fib(n):\n    return n if n < 2 else fib(n-1) + fib(n-2)",
    "with open('test.txt', 'w') as f:\n    f.write('hello')",
]

ppl_before = measure_perplexity(model, tokenizer, VALIDATION_TEXTS)
print(f"Perplexity before injection: {ppl_before:.4f}")

MEMORY_LAYERS = [4, 8, 16, 24]
model, memory_modules = inject_memory_layers(model, MEMORY_LAYERS)

ppl_after = measure_perplexity(model, tokenizer, VALIDATION_TEXTS)
delta_pct = abs(ppl_after - ppl_before) / ppl_before * 100
print(f"Perplexity after injection:  {ppl_after:.4f}")
print(f"Delta: {delta_pct:.2f}%  (must be < 1% to proceed)")

assert delta_pct < 1.0, (
    f"FAIL: perplexity degraded by {delta_pct:.2f}%. "
    "Check gate.bias init (should be -5.0)."
)
print("\nGate OK. Safe to proceed.")

In [ ]:
# ── CELL 6: Download HumanEval + calibration scan ──
# Scans first 100 problems, keeps those with base model pass rate 30–70%.
# Runtime: ~30 min on T4.

from eval import load_humaneval, calibrate_problem_difficulty

raw_problems = load_humaneval("/kaggle/working/problems/humaneval_raw.json")
print(f"Loaded {len(raw_problems)} raw HumanEval problems.")

calibrated = calibrate_problem_difficulty(
    model, tokenizer,
    problems=raw_problems[:100],
    n_attempts=3,
    target_range=(0.3, 0.7),
    save_path="/kaggle/working/problems/calibrated_20.json"
)
print(f"\nCalibrated set: {len(calibrated)} problems saved.")

In [ ]:
# ── CELL 7: Full 10-session evaluation ──
# The main experiment. Runtime: ~3–5 hours on T4.
# Temperature 0.2: allows enough stochasticity for memory to discover
# successful code approaches across sessions.

from eval import run_evaluation

all_results = run_evaluation(
    model, tokenizer, memory_modules,
    n_sessions=10,
    user_id="eval_memory",
    output_dir="/kaggle/working/results/",
    problems_path="/kaggle/working/problems/calibrated_20.json",
    temperature=0.2
)

In [ ]:
# ── CELL 8: Zip checkpoint + results for download ──
# Kaggle sessions expire. Run this BEFORE closing the notebook.

import zipfile

ZIP_PATH = "/kaggle/working/memory_checkpoint.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["memory_states", "results", "problems"]:
        root = f"/kaggle/working/{folder}"
        for dirpath, _, files in os.walk(root):
            for file in files:
                fpath = os.path.join(dirpath, file)
                zf.write(fpath, os.path.relpath(fpath, "/kaggle/working"))

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"Zipped → {ZIP_PATH}  ({size_mb:.1f} MB)")
print("Download via: Output panel → memory_checkpoint.zip → Download")